In [ ]:
import sys
import os
# Path to the folder of the stand-alone model on your machine
path = '/home/noah/message-ix/'
file = 'MESSAGE_CA__test__v4.xlsx'
# Path to subfolder for the stand-alone model data (Excel files)
os.chdir(path)
print(os.getcwd())
sys.path.insert(1, path)


In [ ]:
import message_ix
import ixmp as ix
from ixmp import Scenario


# Stand-alone model
modelName = 'MESSAGEix_CA'      # A MESSAGE model name
scenarioName = 'baseline'       # A MESSAGE model scenario name

mp = ix.Platform()
# mp2 = ix.Platform(dbtype='HSQLDB')

# Notic: either choose I or II
# I) Creating a new MESSAGE stand-alone model
msg = message_ix.Scenario(mp, modelName, scenarioName,
                            version='new', annotation='')


In [ ]:
ssp = 'SSP1'

In [ ]:
sys.path

In [ ]:
scenarios = [   'CD_Links_SSP2_R14|INDC2030i_forever-con-prim-dir-ncr',
                'CD_Links_SSP2_R14|NPi2020_1000-con-prim-dir-ncr']
# for scenario in scenarios:
scenario = scenarios[0]

# Model and scenarios to clone
modelName = scenario.split('|')[0]
scenarioName = scenario.split('|')[1]

# Loading platform
mp1 = ix.Platform(name = 'ene_ixmp')

# Importing the global model scenario used as basis
msgWSC = message_ix.Scenario(mp1, modelName, scenarioName)


In [ ]:

msg.read_excel(path = path + file, init_items = True,
              commit_steps=False)

In [ ]:
from demands import demands
import pandas as pd
import numpy as np
from itertools import product
from downscale_demands import downscale_demands

In [ ]:

nodeName = 'Canada'     # Name of the stand-alone model region.
regionName = 'R14_NAM'      # Name of the reference region in MESSAGE global
ieaName = "'Canada'"         # In the form of: "'Russian Federation'"
plattsName = 'Canada'        # In the form of: 'Russia'
suffix = '_CAN'             # The extension of Excel files, like: '_RU'
iso = 'CAN'
reg = regionName


# region to country mapping
reg_df = pd.read_excel( str(path) +
                       str('/') + str('iso_reg.xlsx'), 'Sheet1')
reg_df['node'] = str('R14_') + reg_df['node'].astype(str)
isoreg = reg_df[reg_df['node']==reg]
isoreg = isoreg['iso'].tolist()

# country and region name mapping
country_map = pd.read_excel( str(path) + str('/') + str(nodeName) +
                            str('_map.xlsx'), 'country_map')
country = country_map.loc[(country_map['include']=='Y')&
                          (country_map['msg_14'] == reg.split('_')[1])]
#    iea = country.loc[country.iso==iso].iea[0]

# SSP GDP data
ssp_gdp_df = pd.read_excel(
        str(path) + str('/') + str('OECD_SSP_GDP_PPP.xlsx'),
        'OECD_SSP_GDP_PPP',
        header = 0 )


ssp_gdp_df = ssp_gdp_df.loc[ ( ssp_gdp_df.Scenario == ssp+ '_v9_130325' )&
                            ( ssp_gdp_df.Region == iso ) ].reset_index(
                                drop=True)

# SSP population data
ssp_pop_df = pd.read_excel(
        str(path) + str('/') + str('OECD_SSP_POP.xlsx'),
        'OECD_SSP_POP',
        header = 0 )

ssp_pop_df = ssp_pop_df.loc[ ( ssp_pop_df.Scenario == ssp + '_v9_130325' )&(
    ssp_pop_df.Region == iso  ) ].reset_index(drop=True)

# SSP data scaling to account for changes since release
wbi_2015_df = pd.read_excel( str(path) + str('/') + str('wbi_pop_gdp_2015.xlsx'), 'wbi_pop_gdp_2015')
wbi_2015_df = wbi_2015_df.loc[(wbi_2015_df.iso == iso)].reset_index(drop=True)
for aaa in ssp_pop_df.index:
    if not (wbi_2015_df.loc[wbi_2015_df['iso']==ssp_pop_df.Region[aaa]]).empty:
        x0 = [ 1e-6*wbi_2015_df.loc[wbi_2015_df['iso']==ssp_pop_df.Region[aaa]].pop_2015.to_numpy()[0] ]
        x = ssp_pop_df.iloc[aaa,range(12,len(ssp_pop_df.columns))]
        dx = x.diff()/ssp_pop_df.iloc[aaa,range(12,len(ssp_pop_df.columns))]
        dx = dx.replace([np.inf,-np.inf,np.nan],0)
        xf = []
        xfi = x0[0]
        for ggg in dx.index:
            rs = xfi*(1+dx[ggg])
            xf.append(round(rs*1e6)/1e6)
            xfi = rs
        ssp_pop_df.iloc[aaa,range(12,len(ssp_pop_df.columns))] = xf


for aaa in ssp_gdp_df.index:
    if not (wbi_2015_df.loc[wbi_2015_df['iso']==ssp_gdp_df.Region[aaa]]).empty:
        x0 = [ 1e-9*wbi_2015_df.loc[wbi_2015_df['iso']==ssp_gdp_df.Region[aaa]].gdp_ppp_usd_2015.to_numpy()[0] ]
        x = ssp_gdp_df.iloc[aaa,range(12,len(ssp_gdp_df.columns))]
        dx = x.diff()/ssp_gdp_df.iloc[aaa,range(12,len(ssp_gdp_df.columns))]
        dx = dx.replace([np.inf,-np.inf,np.nan],0)
        xf = []
        xfi = x0[0]
        for ggg in dx.index:
            rs = xfi*(1+dx[ggg])
            xf.append(round(rs*1e9)/1e9)
            xfi = rs
        ssp_gdp_df.iloc[aaa,range(12,len(ssp_gdp_df.columns))] = xf

    

In [ ]:
ssp_pop_df

In [ ]:

# Get the time parameters
first_year = msg.set('cat_year').loc[msg.set(
        'cat_year')['type_year'] == 'firstmodelyear', 'year'].item()
msg_yrs = [int(x) for x in msg.set('year')]
hist_year = 2015
demand_yrs = [x for x in msg_yrs if x >= hist_year]

df_region = msgWSC.par('demand',{'node': reg,'year':demand_yrs})
df_dem = df_region
cmdtys = df_dem['commodity'].unique().tolist()

# Get the historical demands for country in the region from excel sheet
# xls_par = pd.ExcelFile(path + '/Parameters'+suffix+'.xlsx')
hist_act_df = msg.par('historical_activity').dropna(axis =1)




In [ ]:
df_d = msg.par('demand')
cmdtys = df_d['commodity'].unique().tolist()

In [ ]:
country.iso.values

In [ ]:
# GDP and population
ssp_df = []
for ccc in country.iso.values.tolist():

    c_ssp_gdp_df = ssp_gdp_df.loc[ ssp_gdp_df.Region == ccc ]
    c_ssp_pop_df = ssp_pop_df.loc[ ssp_pop_df.Region == ccc ]
    gdp = []
    pop = []
    gdpc = []

    for jjj in demand_yrs:

        if jjj < 2110:
            mmm = jjj
        else:
            mmm = 2100

        gdp.append(c_ssp_gdp_df[mmm].values[0])
        pop.append(c_ssp_pop_df[mmm].values[0])
        gdpc.append(1e3*c_ssp_gdp_df[mmm].values[0]/c_ssp_pop_df[mmm].values[0])

    ssp_df.append(pd.DataFrame({
        'year':demand_yrs,
        'gdp':gdp,
        'pop':pop,
        'gdpc':gdpc,
        'iso':ccc}))

ssp_df = pd.concat(ssp_df)




In [ ]:
cmdtys

In [ ]:

# initial df for demand data
dmds = []

# Go through each demand commodity and update demands then add to df above
for cmdty in cmdtys:

    # Isolate regional demands
    dff = df_region[df_region['commodity']==cmdty].sort_values('year').reset_index(drop=True)

    # use dff as basis for country df so update node location
    dff['node'] = iso
    dff['node'] = dff['node'].str.replace('PAK','Pakistan')

    # Add percent change
  # dff['pc'] = dff['value'].diff()
    dff['pc'] = dff['value'].pct_change(periods = 1)
  #  for iii in dff.index:
  #       if iii == 0:
  #            dff.pc[iii] = 0
  #    else:
  #           dff.pc[iii] = ( dff.pc[iii] / dff.value[(iii-1)] ) -1

    dff = dff.replace([np.inf,-np.inf,np.nan],0)

    # list of technologies outputing on the demand level
    tec_list = list(set((msg.par(
                    'output', {'level': dff.level.values[0],
                               'commodity': cmdty}))['technology']))

    # filter historical activity datafrme to include technologies for current commodity
    if(len(tec_list)>1):
        ha = hist_act_df.loc[hist_act_df.technology.isin(tec_list)]
    else:
        ha = hist_act_df.loc[hist_act_df['technology'] == tec_list[0]]

    # output efficiency for tecs (typically = 1)
    oe = msg.par('output', {
                        'technology': ha.technology.unique(),
                        'commodity': cmdty}).groupby(['year_act','technology']).mean().reset_index()
    oe = oe[['technology','year_act','value']]

    # update historical activity to account for efficiency losses at the output
    ha = pd.merge( ha, oe, on = ['year_act','technology'], how = 'left' ).dropna()
    ha['value'] = ha['value_x'] * ha['value_y']

    # Go through each demand year and scale the historical demands to follow regional growth pattern
    # must be sorted by year because the state equations are updated sequentially
    for ind in dff.index:

        # if the first year, use the historical demands calibrated to iea db
        if dff.year.iloc[ind] == demand_yrs[0]:

            # isolate the regonal demand target
            # dt = dff.value.iloc[ind]

            # sum iea demands
            dm = ha.loc[ha['year_act']==dff.year.iloc[ind]].groupby('node_loc').sum().reset_index()
            dm = dm[['node_loc','value']]

            # isolate gdp information
            gdp = []
            gdpc = []


            gdp.append(ssp_df.gdp.to_numpy()[0])
            gdpc.append(ssp_df.gdpc.to_numpy()[0])

            dm = pd.DataFrame({'node':dm.node_loc.to_numpy(),
                                'value':dm.value.to_numpy(),
                                'ei':dm.value.to_numpy()/gdp,
                                'gdp':gdp,
                                'gdpc':gdpc})

            dt = dm.value
            dm0 = dm


        # downscale the demands to match growth rate from regional pathway
        else:

            # update total regional demands using growth rate from global model
            dt = dm.value * (1 + dff.pc.iloc[ind])

            # update state equation variables
            dm0 = dm
            gdpf = []
            gdpcf = []

            inds = (ssp_df['year']==dff.year.iloc[ind])
            gdpf.append(ssp_df.loc[inds,'gdp'].to_numpy()[0])
            gdpcf.append(ssp_df.loc[inds,'gdpc'].to_numpy()[0])

            # unload some variables and call LP downscaling algorithm
            names = dm0.node.to_numpy()
            vl = dm0.value.to_numpy()
            gdpco = dm0.gdpc.to_numpy()

            # don't allow 0 intensities in future time steps if not biomass technologies
            if not 'biomass' in cmdty:
                ei = vl / gdpco
               # ei[np.where(ei==0)] = min(ei[ei.nonzero()])
                vl = ei * gdpco

            dmf = downscale_demands(
                dt, # target
                vl, # initial country demands
                gdpco, # initial country income
                gdpcf, # final country income
                gdpf, # final country total GDP
                names) # node names

            # update demand state variables
            dm = pd.DataFrame({'node':dmf.node.to_numpy(),
                                'value':dmf.value.to_numpy(),
                                'ei':dmf.value.to_numpy()/gdpf,
                                'gdp':gdpf,
                                'gdpc':gdpcf})

#         # Check
#         print('--- cmdty ---')
#         print(cmdty)

#         print('--- dm ---')
#         print(dm)

#         print('--- country: total ---')
#         print(dff.value.iloc[ind])

#         print('---country: pc set----')
#         print(dff.pc.iloc[ind])

#         print('---country: pc act----')
#         print(dm.value.sum()/dm0.value.sum()-1)

#         print('--- target ---')
#         print(dt)

#         print('---  total ---')
#         print(dm.value.sum())

        # set the country level demand for this commodity and time step
        dff.value.iloc[ind] = dm.value.to_numpy()[0]

#         if ind > 0 :
#             print('--- diff ---')
#             print(((dff.value.iloc[ind]/dff.value.iloc[ind-1])-1))
#             print(dff.pc.iloc[ind])

    # add demand for particular commodity to the model
    dmds.append(dff[list(msg.par('demand').columns)])

dmds = pd.concat(dmds).reset_index()

In [ ]:
 dmds['node'] = dmds['node'].str.replace('CAN','Canada')

In [ ]:
msg.add_par('demand',dmds)

In [ ]:
msg.commit('demands updated')

In [ ]:
growth = msg.par('growth_activity_up')
growth.head()

In [ ]:
#  msg.check_out()

In [ ]:
# demand = msg.par('demand',{'commodity':'rc_therm'})
# demand.head()

In [ ]:
# msg.remove_par('demand', demand)

In [ ]:
# demand['value'] = demand['value'] * 0.75

In [ ]:
# msg.add_par('demand',demand)

In [ ]:
msg.check_out()

In [ ]:

# growth_gas = growth[growth['technology'].str.contains('gas')]
# msg.remove_par('growth_activity_up', growth_gas)
# growth_gas['value'] = growth_gas['value'] * 0.5
# msg.add_par('growth_activity_up', growth_gas)

In [ ]:
growth_gas_lo = msg.par('growth_activity_lo')
growth_gas_lo = growth_gas_lo[growth_gas_lo['technology'].str.contains('gas')]
msg.remove_par('growth_activity_lo', growth_gas_lo)
growth_gas_lo['value'] = growth_gas_lo['value'] * 3
msg.add_par('growth_activity_lo', growth_gas_lo)

In [ ]:
# #msg.check_out()

# resource_1 = msg.par('resource_volume',{'commodity':'gas_1'})
# resource_2 = msg.par('resource_volume',{'commodity':'gas_2'})
# resource_3 = msg.par('resource_volume',{'commodity':'gas_3'})
# resource_4 = msg.par('resource_volume',{'commodity':'gas_4'})

In [ ]:
# frames = [resource_1, resource_2, resource_3, resource_4]
# import pandas as pd
# resource = pd.concat(frames)

In [ ]:
# resource

In [ ]:
# #resource['value'] = resource['value'] *2.5 
# msg.add_par('resource_volume', resource)


In [ ]:
msg.model

In [ ]:
msg.version

In [ ]:
msg.commit('')

In [ ]:

# Exporting the built model (Scenario) to GAMS with an optional case name
caseName = msg.model + '__' + msg.scenario + '__v' + str(msg.version)

# Solving the model
msg.solve(model='MESSAGE', case=caseName)


In [ ]:
nodeloc = 'Canada'
from postprocess import plotdf_sec, plotdf, group, multiply_df
yr = msg.set('cat_year', {'type_year': 'firstmodelyear'}).year.item()

Now making another scenario cloned from baseline to add other policies. 

Policy No. 1
Baseline - NDC 
Implement Canada's commitments 
msg = msg.clone(keep_solution=False)

In [ ]:
# A function for making model output ready
def model_output(msgBAS, tecs, nodeloc, parname, coms=None):
    df1 = msg.var('ACT', {'technology': tecs, 'node_loc': nodeloc})
    df2 = msg.par(parname, {'technology': tecs, 'node_loc': nodeloc})
    if coms:
        if isinstance(coms, str):
            coms = [coms]
        df2 = df2.loc[df2['commodity'].isin(coms)]
    df = multiply_df(df1, 'lvl', df2, 'value')
    return df, df2


In [ ]:

tecs = list(set(msg.par('output', {'level': 'primary'}).technology))
tecs_imp = list(['coal_imp', 'oil_imp', 'gas_imp', 'u5_imp', 'meth_imp',
                 'loil_imp', 'LNG_imp', 'lh2_imp', 'foil_imp', 'eth_imp',
                 'elec_imp'])
# TODO: export must be deducted from import and PE
tecs_exp = [x.split('_')[0] + '_exp' for x in tecs_imp if
            x.split('_')[0] + '_exp' in list(msg.set('technology'))]

df, df2 = model_output(msg, tecs + tecs_imp, nodeloc, 'output')
df = group(df, ['year_act', 'commodity'], 'product', 0.0, yr)
df = df.reindex(['crudeoil', 'biomass', 'electr', 'gas', 'lightoil',
                 'coal', 'fueloil', 'u5', 'rc_therm', 'others'], axis=1)


In [ ]:
primary = report_df.filter(variable = 'Primary Energy|crudeoil|*')

In [ ]:
from message_ix.reporting import Reporter
from update_units import unit_correction
import numpy as np
unit_correct = True  # unit s must be corrected to run this script
from_excel = False
min_year = 2015
max_year = 2060

In [ ]:
from message_ix.reporting import Reporter
from update_units import unit_correction
import numpy as np
unit_correct = True  # unit s must be corrected to run this script
from_excel = False
min_year = 2015
max_year = 2060

if unit_correct:
    msg = msg.clone(keep_solution=False)
    unit_correction(mp, msg)
    caseName = msg.model + '__' + msg.scenario + '__v' + str(msg.version)
    # Solving the model
    msg.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4'})

##%% Generating reporter
rep = Reporter.from_scenario(msg)
report = rep.get('message:default') # in fact, only in = input x ACT, out = output x ACT is needed

if from_excel:
    # Read default report
    report_df = pd.read_csv('MESSAGE_NO_example_default_report.csv')
else:
# Or if trying another country model, execute the next 4:
    report_df = report.timeseries()
    report_df.reset_index(inplace=True)
    report_df.columns = report_df.columns.astype(str)
    report_df.columns = report_df.columns.str.title()


In [ ]:
# Plotted years
years = msg.set('year').tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

# Create data for Sankey diagrams and plot it using plotly
report_df.fillna(0, inplace = True)

# Convert from GWa to TWh (*8.76)
c = report_df.select_dtypes(include=[np.number]) * 8.76
report_df[c.columns] = c

# Drop not necessary years
report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(report_df.columns.to_series()['2020': str(min_year)],
                           axis=1)

# Identify input and output variables and its flows (provided within the Reporter: in = input x ACT, out = output x ACT)
in_flows = report_df[report_df['Variable'].str.startswith("in|")]
out_flows = report_df[report_df['Variable'].str.startswith("out|")]
all_flows = out_flows.append(in_flows)

In [ ]:
years

In [ ]:
report_df.to_csv('MESSAGE_CANADA.csv')

In [ ]:
report_df

In [ ]:
report_df["Variable"] = report_df["Variable"].str.replace('emis', 'Emissions', regex = False)
#report_df["Variable"] = report_df["Variable"].str.replace('in|final|', 'Final Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|final|', 'Final Energy|', regex = False)
#report_df["Variable"] = report_df["Variable"].str.replace('in|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('total om cost|', 'Total Costs|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|useful|', 'Useful Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|useful|', 'Useful Energy|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('emis', 'Emissions', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('in|final|', 'Final Energy|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('in|primary|', 'Primary Energy|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('total om cost|', 'Total Costs|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('in|renewable|', 'Primary Energy|renewable|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('out|useful|', 'Useful Energy|', regex = False)

In [ ]:
import pyam
import matplotlib.pyplot as plt
import pyam
plotyrs = [2020,2025,2030,2035,2040,2045,2050,2055,2060]
pyam_df = pyam.IamDataFrame(report_df)
pdf = pyam_df.copy()

In [ ]:

biomass = ['Primary Energy|biomass|bio_istig_ccs|M1', 
'Primary Energy|biomass|bio_istig|M1']
#'Primary Energy|biomass|bio_ppl|M1',
#'Primary Energy|biomass|biomass_nc|M1',
#'Primary Energy|biomass|biomass_t_d|M1',
#'Primary Energy|biomass|eth_bio_ccs|M1',
#'Primary Energy|biomass|eth_bio|M1',
#'Primary Energy|biomass|gas_bio|M1',
#'Primary Energy|biomass|h2_bio_ccs|M1',
#'Primary Energy|biomass|h2_bio|M1',
#'Primary Energy|biomass|liq_bio_ccs|M1',
#'Primary Energy|biomass|liq_bio|M1','Primary Energy|biomass|land_use_biomass|M1']

lng = 'Primary Energy|LNG|LNG_bal|M1'

coal = ['Primary Energy|coal|coal_bal|M1',
                      'Primary Energy|coal|coal_exp|M1',
                      'Primary Energy|coal|coal_exp|M1',
                      'Primary Energy|coal|coal_extr_ch4|M1',
                      'Primary Energy|coal|coal_extr|M1',
                      'Primary Energy|coal|lignite_extr|M1']
gas = [#'Primary Energy|gas|gas_bal|M1',
    'Primary Energy|gas|gas_extr_1|M1',
      'Primary Energy|gas|gas_extr_2|M1',
      'Primary Energy|gas|gas_extr_3|M1',
      'Primary Energy|gas|gas_extr_4|M1',
      'Primary Energy|gas|gas_extr_5|M1',
      'Primary Energy|gas|gas_extr_6|M1']
      #'Primary Energy|gas|LNG_prod|M1']
      # 'Primary Energy|LNG|LNG_bal|M1']

oil = [#'Primary Energy|crudeoil|oil_bal|M1',
    'Primary Energy|crudeoil|oil_exp|M1',
     # 'Primary Energy|crudeoil|oil_extr_1_ch4|M1',
        'Primary Energy|crudeoil|oil_extr_1|M1',
       # 'Primary Energy|crudeoil|oil_extr_2_ch4|M1',
        'Primary Energy|crudeoil|oil_extr_2|M1',
      #  'Primary Energy|crudeoil|oil_extr_3_ch4|M1',
        'Primary Energy|crudeoil|oil_extr_3|M1',
        #'Primary Energy|crudeoil|oil_extr_4_ch4|M1',
        'Primary Energy|crudeoil|oil_extr_4|M1',
        'Primary Energy|crudeoil|oil_extr_5|M1',
        'Primary Energy|crudeoil|oil_extr_6|M1',
        'Primary Energy|crudeoil|oil_extr_7|M1']

hydro = ['Primary Energy|renewable|hydro|hydro_hc|M1',  'Primary Energy|renewable|hydro|hydro_lc|M1']

biomass_renew = ['Primary Energy|renewable|biomass|bio_extr_a|M1',
                'Primary Energy|renewable|biomass|bio_extr_b|M1',
                'Primary Energy|renewable|biomass|bio_extr_c|M1',
                'Primary Energy|renewable|biomass|bio_extr_d|M1',
                'Primary Energy|renewable|biomass|bio_extr_e|M1',
                'Primary Energy|renewable|biomass|bio_extr_f|M1',
                'Primary Energy|renewable|biomass|bio_extr_g|M1']

solar = ['Primary Energy|renewable|solar_pv|solar_pv_ppl|M1' , 
         'Primary Energy|renewable|solar_th|solar_th_ppl|M1']


wind = ['Primary Energy|renewable|wind|wind_ppf|M1', 'Primary Energy|renewable|wind|wind_ppl|M1']

lh2 = ['Primary Energy|lh2|lh2_bal|M1', 'Primary Energy|lh2|lh2_exp|M1']

ethanol = ['Primary Energy|methanol|meth_bal|M1',  'Primary Energy|methanol|meth_exp|M1']

methanol = ['Primary Energy|ethanol|eth_bal|M1', 'Primary Energy|ethanol|eth_exp|M1']



In [ ]:
pdf = pyam_df.copy()


In [ ]:
pdf.aggregate(
    "Primary Energy|coal",
    components=coal,
    append=True
)


pdf.aggregate(
    "Primary Energy|renewable|solar",
    components=solar,
    append=True
)

pdf.aggregate(
    "Primary Energy|renewable|hydro",
    components=hydro,
    append=True
)

pdf.aggregate(
    "Primary Energy|ethanol",
    components=ethanol,
    append=True
)
pdf.aggregate(
    "Primary Energy|methanol",
    components=methanol,
    append=True
)
pdf.aggregate(
    "Primary Energy|Liq. Hydrogen",
    components=lh2,
    append=True
)
pdf.aggregate(
    "Primary Energy|renewable|wind",
    components=wind,
    append=True
)

pdf.aggregate(
    "Primary Energy|renewable|biomass",
    components= biomass_renew,
    append=True
)


pdf.aggregate(
    "Primary Energy|biomass",
    components= biomass,
    append=True
)

pdf.aggregate(
    "Primary Energy|oil",
    components=oil,
    append=True
)

pdf.aggregate(
    "Primary Energy|gas",
    components=gas,
    append=True
)

pdf.aggregate(
    "Primary Energy|LNG",
    components=lng,
    append=True
)

In [ ]:
pdf.filter(variable = 'Primary Energy|oil').timeseries()

In [ ]:
pdf.filter(scenario = 'baseline', variable=["Primary Energy|renewable|solar",
                                            #"Primary Energy|LNG",
                                            "Primary Energy|gas",
                                            "Primary Energy|oil",
                                            "Primary Energy|biomass",
                                            "Primary Energy|renewable|wind",
                                            "Primary Energy|Liq. Hydrogen",
                                            "Primary Energy|methanol",
                                            "Primary Energy|ethanol",
                                            "Primary Energy|renewable|hydro",
                                            "Primary Energy|renewable|biomass",
                                            "Primary Energy|methanol",
                                           "Primary Energy|coal"],
              year = plotyrs).stack_plot(total=True,
                                        title = 'Primary Energy by fuel - Baseline Scenario')

In [ ]:
# gas = pyam_df.filter(scenario = 'baseline', variable=[
#                                             "Primary Energy|gas",
#                                             ],
#               year = plotyrs)

In [ ]:
gas.head(10)

In [ ]:
i_feed = ['Useful Energy|i_feed|ethanol_fs|M1', 'Useful Energy|i_feed|gas_fs|M1',
         'Useful Energy|i_feed|loil_fs|M1', 'Useful Energy|i_feed|methanol_fs|M1']

transport = ['Useful Energy|transport|coal_trp|M1',
            'Useful Energy|transport|elec_trp|M1',
            'Useful Energy|transport|eth_fc_trp|M1',
            'Useful Energy|transport|eth_ic_trp|M1',
            'Useful Energy|transport|foil_trp|M1',
            'Useful Energy|transport|gas_trp|M1',
            'Useful Energy|transport|h2_fc_trp|M1',
            'Useful Energy|transport|loil_trp|M1',
            'Useful Energy|transport|meth_fc_trp|M1',
            'Useful Energy|transport|meth_ic_trp|M1']


i_spec = ['Useful Energy|i_spec|h2_fc_I|M1',
          'Useful Energy|i_spec|sp_coal_I|M1',
          'Useful Energy|i_spec|sp_el_I|M1',
          'Useful Energy|i_spec|sp_eth_I|M1',
          'Useful Energy|i_spec|sp_liq_I|M1',
          'Useful Energy|i_spec|sp_meth_I|M1']

i_therm =   ['Useful Energy|i_therm|biomass_i|M1',
            'Useful Energy|i_therm|coal_i|M1',
            'Useful Energy|i_therm|elec_i|M1',
            'Useful Energy|i_therm|eth_i|M1',
            'Useful Energy|i_therm|foil_i|M1',
            'Useful Energy|i_therm|gas_i|M1',
            'Useful Energy|i_therm|h2_fc_I|M1',
            'Useful Energy|i_therm|h2_i|M1',
            'Useful Energy|i_therm|heat_i|M1',
            'Useful Energy|i_therm|hp_el_i|M1',
            'Useful Energy|i_therm|hp_gas_i|M1',
            'Useful Energy|i_therm|loil_i|M1',
            'Useful Energy|i_therm|meth_i|M1',
            'Useful Energy|i_therm|solar_i|M1']
rc_spec = ['Useful Energy|rc_spec|h2_fc_RC|M1',
          'Useful Energy|rc_spec|sp_el_RC|M1']

rc_therm = ['Useful Energy|rc_therm|biomass_rc|M1',
            'Useful Energy|rc_therm|coal_rc|M1',
            'Useful Energy|rc_therm|eth_rc|M1',
            'Useful Energy|rc_therm|foil_rc|M1',
            'Useful Energy|rc_therm|gas_rc|M1',
            'Useful Energy|rc_therm|h2_fc_RC|M1',
            'Useful Energy|rc_therm|h2_rc|M1',
            'Useful Energy|rc_therm|heat_rc|M1',
            'Useful Energy|rc_therm|hp_el_rc|M1',
            'Useful Energy|rc_therm|hp_gas_rc|M1',
            'Useful Energy|rc_therm|loil_rc|M1',
            'Useful Energy|rc_therm|meth_rc|M1',
            'Useful Energy|rc_therm|solar_rc|M1']

biom = ['Useful Energy|non-comm|biomass_nc|M1']

In [ ]:
df_useful = pyam_df.copy()
# df_useful.aggregate(
#     "Useful Energy|non-Commercial biomass",
#     components=biom,
#     append=True
# )

df_useful.aggregate(
    "Useful Energy|residential/commercial thermal",
    components=rc_therm,
    append=True
)

df_useful.aggregate(
    "Useful Energy|residential/commercial specific",
    components=rc_spec,
    append=True
)

df_useful.aggregate(
    "Useful Energy|industrial thermal",
    components=i_therm,
    append=True
)

df_useful.aggregate(
    "Useful Energy|industrial specific",
    components=i_spec,
    append=True
)

df_useful.aggregate(
    "Useful Energy|industrial feedstock",
    components=i_feed,
    append=True
)

df_useful.aggregate(
    "Useful Energy|transport",
    components=i_therm,
    append=True
)


In [ ]:
df_useful.filter(scenario = 'baseline', variable=[
    "Useful Energy|non-Commercial biomass",
    "Useful Energy|residential/commercial thermal",
    "Useful Energy|residential/commercial specific",
    "Useful Energy|industrial thermal",
    "Useful Energy|industrial specific",
    "Useful Energy|industrial feedstock",
    "Useful Energy|transport"],
              year = plotyrs).stack_plot(total=True,
                                         title = 'Demands by sector')

In [ ]:
cement = ['Emissions|CO2|cement_CO2|M1']
coal = ['Emissions|CO2|coal_extr_ch4|M1',
        'Emissions|CO2|coal_extr|M1',
        'Emissions|CO2|coal_imp|M1']
foil = ['Emissions|CO2|foil_imp|M1']
gas = ['Emissions|CO2|gas_extr_1|M1',
        'Emissions|CO2|gas_extr_2|M1',
        'Emissions|CO2|gas_extr_3|M1',
        'Emissions|CO2|gas_extr_4|M1',
        'Emissions|CO2|gas_extr_5|M1',
        'Emissions|CO2|gas_extr_6|M1'
        ]
lignite = ['Emissions|CO2|lignite_extr|M1']
loil = ['Emissions|CO2|loil_imp|M1']
meth_ccs = ['Emissions|CO2|meth_coal_ccs|M1']
oil = ['Emissions|CO2|oil_extr_1_ch4|M1',
    'Emissions|CO2|oil_extr_1|M1',
    'Emissions|CO2|oil_extr_2_ch4|M1',
    'Emissions|CO2|oil_extr_2|M1',
    'Emissions|CO2|oil_extr_3_ch4|M1',
    'Emissions|CO2|oil_extr_3|M1',
    'Emissions|CO2|oil_extr_4_ch4|M1',
    'Emissions|CO2|oil_extr_4|M1',
    'Emissions|CO2|oil_extr_5|M1',
    'Emissions|CO2|oil_extr_6|M1',
    'Emissions|CO2|oil_extr_7|M1']

oil_trade = ['Emissions|CO2|foil_imp|M1',
             'Emissions|CO2|loil_exp|M1',
            'Emissions|CO2|loil_imp|M1',
             'Emissions|CO2|oil_imp|M1']
       
gas_trade = ['Emissions|CO2|LNG_exp|M1',
            'Emissions|CO2|LNG_imp|M1',
            'Emissions|CO2|gas_imp|M1']
            #'Emissions|CO2|LNG_exp|M1']

In [ ]:
df_emis_b = pyam_df.filter(variable = 'Emissions|CO2|*',year = plotyrs)

In [ ]:
df_emis_b.to_csv('emissions.csv')

In [ ]:
df_emis = pyam_df.copy()


df_emis.aggregate(
    "Emissions|CO2|gas",
    components=gas,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|oil",
    components=oil,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|coal",
    components=coal,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|cement",
    components=cement,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|oil trade",
    components=oil_trade,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|gas trade",
    components=gas_trade,
    append=True
)

In [ ]:
df_emis.filter(scenario = 'baseline', variable=
    ["Emissions|CO2|gas",
    "Emissions|CO2|oil",
    "Emissions|CO2|coal",
    "Emissions|CO2|cement",
    "Emissions|CO2|oil trade",
    "Emissions|CO2|gas trade"],
              year = plotyrs).stack_plot(total=True,
                                         title = 'CO2 Emissions - Baseline Scenario')

In [ ]:
df_emis.aggregate('Emissions|*').timeseries()

In [ ]:

coal = ['Total Costs|coal_adv',
        'Total Costs|coal_adv_ccs',
        'Total Costs|coal_bal',
        'Total Costs|coal_exp',
        'Total Costs|coal_extr',
        'Total Costs|coal_extr_ch4',
        'Total Costs|coal_gas',
        'Total Costs|coal_hpl',
        'Total Costs|coal_i',
        'Total Costs|coal_imp',
        'Total Costs|coal_ppl',
        'Total Costs|coal_ppl_u',
        'Total Costs|coal_rc',
        'Total Costs|coal_ppl',
        'Total Costs|coal_ppl_u',
       'Total Costs|coal_t_d',
       'Total Costs|syn_liq',
        'Total Costs|meth_coal',
       'Total Costs|lignite_extr']
biomass = ['bio_hpl', 'bio_ppl',
          'Total Costs|liq_bio']


elec = ['Total Costs|elec_exp',
        'Total Costs|elec_i',
        'Total Costs|elec_imp',
        'Total Costs|elec_t_d',
        'Total Costs|elec_trp']

oil = ['Total Costs|foil_i',
    'Total Costs|foil_imp',
    'Total Costs|foil_ppl',
    'Total Costs|oil_extr_1'
    'Total Costs|oil_extr_1_ch4'
    'Total Costs|oil_extr_2',
    'Total Costs|oil_extr_3',
    'Total Costs|oil_extr_4',
    'Total Costs|oil_extr_5',
    'Total Costs|oil_extr_6',
    'Total Costs|oil_extr_7',
    'Total Costs|oil_extr_2_ch4',
    'Total Costs|oil_extr_3_ch4',
    'Total Costs|loil_cc',
    'Total Costs|loil_ppl']
gas = ['Total Costs|gas_bal',
    'Total Costs|gas_bio',
    'Total Costs|gas_cc',
    'Total Costs|gas_cc_ccs',
    'Total Costs|gas_ct',
    'Total Costs|gas_extr_1',
    'Total Costs|gas_extr_2',
    'Total Costs|gas_extr_3',
    'Total Costs|gas_extr_4',
    'Total Costs|gas_extr_5',
    'Total Costs|gas_extr_6',
    'Total Costs|gas_extr_mpen',
    'Total Costs|gas_fs',
    'Total Costs|gas_hpl',
    'Total Costs|gas_i',
    'Total Costs|gas_imp',
    'Total Costs|gas_ppl',
    'Total Costs|gas_rc',
    'Total Costs|gas_t_d',
    'Total Costs|gas_t_d_ch4',
    'Total Costs|gas_trp',
      'Total Costs|igcc']

geo_ppl = ['Total Costs|geo_hpl','Total Costs|geo_ppl']

solar = ['Total Costs|solar_i',
        'Total Costs|solar_pv_ppl',
        'Total Costs|solar_rc',
        'Total Costs|solar_th_ppl']

wind = ['Total Costs|wind_curtailment1',
'Total Costs|wind_curtailment2',
'Total Costs|wind_curtailment3',
'Total Costs|wind_ppf',
'Total Costs|wind_ppl']

hydro = ['Total Costs|hydro_hc',
'Total Costs|hydro_lc']

nuclear = ['Total Costs|nuc_lc']


In [ ]:
df_cost = pyam_df.copy()


df_cost.aggregate(
    "Total Costs|coal",
    components=coal,
    append=True
)

df_cost.aggregate(
    "Total Costs|biomass",
    components=biomass,
    append=True
)

df_cost.aggregate(
    "Total Costs|electricity",
    components=elec,
    append=True
)

df_cost.aggregate(
    "Total Costs|oil",
    components=biomass,
    append=True
)

df_cost.aggregate(
    "Total Costs|gas",
    components=gas,
    append=True
)

df_cost.aggregate(
    "Total Costs|geothermal",
    components=geo_ppl,
    append=True
)

df_cost.aggregate(
    "Total Costs|solar",
    components=solar,
    append=True
)

df_cost.aggregate(
    "Total Costs|wind",
    components=wind,
    append=True
)

df_cost.aggregate(
    "Total Costs|hydro",
    components=hydro,
    append=True
)
df_cost.aggregate(
    "Total Costs|nuclear",
    components=nuclear,
    append=True
)

In [ ]:
df_cost.filter(scenario = 'baseline', variable=
    ["Total Costs|coal","Total Costs|biomass",
    "Total Costs|electricity",
    "Total Costs|oil",
    "Total Costs|gas",
    "Total Costs|geothermal",
    "Total Costs|solar",
    "Total Costs|wind",
    "Total Costs|hydro",
    "Total Costs|nuclear"],
              year = plotyrs).stack_plot(total=True,
                                         title = 'Total (Operational+ Investment) Costs - Baseline')


Now making another scenario cloned from baseline to add other policies. 

Policy No. 1
Baseline - NDC 
Implement Canada's commitments 


In [ ]:
msgbas = msg.clone(modelName, 'Baseline-NDC','Introducing Canadas goals',
                     keep_solution=False)

In [ ]:
msgbas.check_out()

In [ ]:
msg.par('emiss')

In [ ]:

def expand_grid(dictionary):
    return pd.DataFrame([row for row in product(*dictionary.values())], 
                       columns=dictionary.keys())


In [ ]:
  
    
year_df = [2020, 2025, 2030, 2035, 2040 ,2045 ,2050 ,2055 ,2060 ]


dic = {
    'node': [nodeName],
    'type_emission':[str('TCE')],
    'type_tec':[str('all')],
    'type_year': [2020, 2025, 2030, 2035, 2040 ,2045 ,2050 ,2055 ,2060 ],
    'value':     [629, 572, 526, 450 , 375, 320, 255, 200, 150],
    'unit':[str('-')]}

df = expand_grid(dic)
df = df.iloc[0:9]
df['type_year'] = year_df



In [ ]:
# msgbas.remove_solution()

In [ ]:
msgbas.check_out()

In [ ]:
#msgSDG.add_par('bound_emission',df)
msgbas.add_par('bound_emission', df)
msgbas.commit('emissions bounded ')

In [ ]:
msgbas.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4'})

In [ ]:
from message_ix.reporting import Reporter
from update_units import unit_correction
import numpy as np
unit_correct = False  # unit s must be corrected to run this script
from_excel = False
min_year = 2015
max_year = 2060

if unit_correct:
    msg = msg.clone(keep_solution=False)
    unit_correction(mp, msg)
    caseName = msg.model + '__' + msg.scenario + '__v' + str(msg.version)
    # Solving the model
    msg.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4'})

##%% Generating reporter
rep = Reporter.from_scenario(msgbas)
report = rep.get('message:default') # in fact, only in = input x ACT, out = output x ACT is needed

if from_excel:
    # Read default report
    report_df = pd.read_csv('MESSAGE_NO_example_default_report.csv')
else:
# Or if trying another country model, execute the next 4:
    report_df = report.timeseries()
    report_df.reset_index(inplace=True)
    report_df.columns = report_df.columns.astype(str)
    report_df.columns = report_df.columns.str.title()

    # Plotted years
years = msgbas.set('year').tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

# Create data for Sankey diagrams and plot it using plotly
report_df.fillna(0, inplace = True)

# Convert from GWa to TWh (*8.76)
c = report_df.select_dtypes(include=[np.number]) * 8.76
report_df[c.columns] = c

# Drop not necessary years
report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(report_df.columns.to_series()['2020': str(min_year)],
                           axis=1)

# Identify input and output variables and its flows (provided within the Reporter: in = input x ACT, out = output x ACT)
in_flows = report_df[report_df['Variable'].str.startswith("in|")]
out_flows = report_df[report_df['Variable'].str.startswith("out|")]
all_flows = out_flows.append(in_flows)

In [ ]:
report_df.to_csv('MESSAGE_CANADA_baselinendc.csv')

In [ ]:
report_df["Variable"] = report_df["Variable"].str.replace('emis', 'Emissions', regex = False)
#report_df["Variable"] = report_df["Variable"].str.replace('in|final|', 'Final Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|final|', 'Final Energy|', regex = False)
#report_df["Variable"] = report_df["Variable"].str.replace('in|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('total om cost|', 'Total Costs|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|useful|', 'Useful Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|useful|', 'Useful Energy|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('emis', 'Emissions', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('in|final|', 'Final Energy|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('in|primary|', 'Primary Energy|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('total om cost|', 'Total Costs|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('in|renewable|', 'Primary Energy|renewable|', regex = False)
# report_df1["Variable"] = report_df1["Variable"].str.replace('out|useful|', 'Useful Energy|', regex = False)

In [ ]:
import pyam
import matplotlib.pyplot as plt
import pyam
plotyrs = [2020,2025,2030,2035,2040,2045,2050,2055,2060]
pyam_df = pyam.IamDataFrame(report_df)
pdf = pyam_df.copy()

In [ ]:
pdf.aggregate(
    "Primary Energy|coal",
    components=coal,
    append=True
)


pdf.aggregate(
    "Primary Energy|renewable|solar",
    components=solar,
    append=True
)

pdf.aggregate(
    "Primary Energy|renewable|hydro",
    components=hydro,
    append=True
)

pdf.aggregate(
    "Primary Energy|ethanol",
    components=ethanol,
    append=True
)
pdf.aggregate(
    "Primary Energy|methanol",
    components=methanol,
    append=True
)
pdf.aggregate(
    "Primary Energy|Liq. Hydrogen",
    components=lh2,
    append=True
)
pdf.aggregate(
    "Primary Energy|renewable|wind",
    components=wind,
    append=True
)

pdf.aggregate(
    "Primary Energy|renewable|biomass",
    components= biomass_renew,
    append=True
)


pdf.aggregate(
    "Primary Energy|biomass",
    components= biomass,
    append=True
)

pdf.aggregate(
    "Primary Energy|oil",
    components=oil,
    append=True
)

pdf.aggregate(
    "Primary Energy|gas",
    components=gas,
    append=True
)

pdf.aggregate(
    "Primary Energy|LNG",
    components=lng,
    append=True
)

In [ ]:
pdf.filter(variable = 'Primary Energy|oil').timeseries()

In [ ]:
pdf.filter( variable=["Primary Energy|renewable|solar",
                                            #"Primary Energy|LNG",
                                            "Primary Energy|gas",
                                            "Primary Energy|oil",
                                            "Primary Energy|biomass",
                                            "Primary Energy|renewable|wind",
                                            "Primary Energy|Liq. Hydrogen",
                                            "Primary Energy|methanol",
                                            "Primary Energy|ethanol",
                                            "Primary Energy|renewable|hydro",
                                            "Primary Energy|renewable|biomass",
                                            "Primary Energy|methanol",
                                           "Primary Energy|coal"],
              year = plotyrs).stack_plot(total=True,
                                        title = 'Primary Energy by fuel - Baseline Scenario')

In [ ]:
df_emis = pyam_df.copy()


df_emis.aggregate(
    "Emissions|CO2|gas",
    components=gas,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|oil",
    components=oil,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|coal",
    components=coal,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|cement",
    components=cement,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|oil trade",
    components=oil_trade,
    append=True
)

df_emis.aggregate(
    "Emissions|CO2|gas trade",
    components=gas_trade,
    append=True
)

In [ ]:
df_emis.timeseries()

In [ ]:
df_emis.filter( variable=
    ["Emissions|CO2|gas",
    "Emissions|CO2|oil",
    "Emissions|CO2|coal",
    "Emissions|CO2|cement",
    "Emissions|CO2|oil trade",
    "Emissions|CO2|gas trade"],
              year = plotyrs,region = 'Canada').stack_plot(total=True,
                                         title = 'CO2 Emissions - Baseline Scenario')

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('MESSAGE_CANADA_baselinendc.csv')

In [ ]:
df['Model'].replace('CD_Links_SSP2_R14', 'MESSAGEix-Canada', inplace=True)

In [ ]:
df.head()

In [ ]:
df.to_csv('MESSAGE_CANADA_v1.csv')

In [ ]:
import pandas as pd

In [ ]:
iso = 'CAN'

In [ ]:
# SSP GDP data
ssp_gdp_df = pd.read_excel( str('OECD_SSP_GDP_PPP.xlsx'),
        'OECD_SSP_GDP_PPP',
        header = 0)
ssp_gdp_df = ssp_gdp_df.loc[ (  ssp_gdp_df.Region == iso ) ].reset_index(drop=True)

# SSP population data

ssp_pop_df = pd.read_excel(str('OECD_SSP_POP.xlsx'),
        'OECD_SSP_POP',
        header = 0)
ssp_pop_df = ssp_pop_df.loc[ (  ssp_pop_df.Region == iso  ) ].reset_index(drop=True)

In [ ]:
ssp_pop_df['Scenario'] = ssp_pop_df['Scenario'].str.replace('_v9_130325', '')
ssp_gdp_df['Scenario'] = ssp_gdp_df['Scenario'].str.replace('_v9_130325', '')

In [ ]:
ssp_gdp_df

In [ ]:
import pyam
import matplotlib.pyplot as plt
import pyam

pop_df = pyam.IamDataFrame(ssp_pop_df)
gdp_df = pyam.IamDataFrame(ssp_gdp_df)

In [ ]:
pop_df.head()

In [ ]:
pop_df.plot(color='scenario', title='Population Projections of Canada across SSPs')

In [ ]:
gdp_df.plot(color='scenario', title='GDP Projections of Canada across SSPs')